# 08b — Balance Dataset & Run AutoML (CSV-driven)

**Same as notebook 08 but simpler:** uses `combined_dataset.csv` as the single source of truth.
Downloads all images from their original sources, augments types 4/5/6, uploads once to GCS, runs AutoML.

No bucket caching, no Drive restore — just CSV → download → augment → upload → AutoML.

**Augmentation (Albumentations):** flips, rotations, random crop, Gaussian noise
**Forbidden:** brightness, contrast, color jitter, blurring

In [ ]:
# Cell 1: Setup
import os, subprocess, sys

if not os.path.exists("/content/NST_Class"):
    subprocess.run(["git", "clone", "https://github.com/AayushBaniya2006/NST_Class.git"], cwd="/content")
else:
    subprocess.run(["git", "pull"], cwd="/content/NST_Class")
os.chdir("/content/NST_Class")

!pip install -q -r requirements.txt
!pip install -q albumentations

# Download Fitzpatrick17k CSV (has URLs for image download)
os.makedirs("data", exist_ok=True)
if not os.path.exists("data/fitzpatrick17k.csv"):
    !wget -q -O data/fitzpatrick17k.csv "https://raw.githubusercontent.com/mattgroh/fitzpatrick17k/main/fitzpatrick17k.csv"
    print("Downloaded fitzpatrick17k.csv")

sys.path.insert(0, "/content/NST_Class")
print("Setup complete!")

In [ ]:
# Cell 2: Authenticate with Google Cloud
from google.colab import auth
auth.authenticate_user()
print("Authenticated!")

In [ ]:
# Cell 3: Load the CSV — single source of truth
import pandas as pd
import numpy as np
from pathlib import Path
import shutil, time

BUCKET_NAME = "skin-tone-project"
GCS_IMAGE_PREFIX = f"gs://{BUCKET_NAME}/combined_images"
LOCAL_IMAGE_DIR = "data/combined_images"
LOCAL_AUG_DIR = "data/augmented_images"
SEED = 42
IMAGE_SIZE = 224

AUGMENTATION_TARGETS = {
    4: 1000,   # 2464 + 1000 = 3464
    5: 2000,   # 1258 + 2000 = 3258
    6: 2500,   # 492 + 2500 = 2992
}

df = pd.read_csv("combined_dataset.csv")
print("Combined dataset (from CSV):")
print(df["fitzpatrick_scale"].value_counts().sort_index())
print(f"\nTotal: {len(df)}")
print(f"\nBy source:")
print(df["source"].value_counts())
print(f"\nAugmentation plan:")
for cls, count in sorted(AUGMENTATION_TARGETS.items()):
    current = len(df[df["fitzpatrick_scale"] == cls])
    print(f"  Type {cls}: {current} → {current + count} (+{count} augmented)")

In [ ]:
# Cell 4: Download ALL images from original sources
# Uses combined_dataset.csv to know exactly what to download
os.makedirs(LOCAL_IMAGE_DIR, exist_ok=True)

existing = {f.name for f in Path(LOCAL_IMAGE_DIR).iterdir() if f.is_file()}
needed = set(df["img_id"]) - existing
print(f"Already have: {len(existing)}, need: {len(needed)}")

# ---- FITZPATRICK17K: download from URLs ----
fitz_ids = set(df[df["source"] == "fitzpatrick17k"]["img_id"])
fitz_needed = needed & fitz_ids
if fitz_needed:
    print(f"\n--- Fitzpatrick17k: downloading {len(fitz_needed)} images from URLs ---")
    fitz_csv = pd.read_csv("data/fitzpatrick17k.csv")
    fitz_csv["img_id"] = fitz_csv["md5hash"] + ".jpg"
    to_download = fitz_csv[fitz_csv["img_id"].isin(fitz_needed)].copy()
    to_download = to_download.rename(columns={"md5hash": "hasher"})

    from src.data.prepare import download_images
    t0 = time.time()
    count = download_images(to_download, LOCAL_IMAGE_DIR, max_workers=50, batch_size=1000)
    elapsed = time.time() - t0
    print(f"Fitzpatrick17k: {count}/{len(fitz_needed)} downloaded in {elapsed:.0f}s")
    if count < len(fitz_needed):
        print(f"  {len(fitz_needed) - count} URLs are dead (expected for this dataset)")
else:
    print("Fitzpatrick17k: all present")

# ---- SCIN: download from public GCS bucket ----
scin_ids = set(df[df["source"] == "scin"]["img_id"])
scin_needed = (needed - existing) & scin_ids  # re-check after fitz downloads
# Re-check existing
existing = {f.name for f in Path(LOCAL_IMAGE_DIR).iterdir() if f.is_file()}
scin_needed = scin_ids - existing
if scin_needed:
    print(f"\n--- SCIN: downloading {len(scin_needed)} images from public bucket ---")
    os.environ["CLOUDSDK_STORAGE_THREAD_COUNT"] = "16"
    os.environ["CLOUDSDK_STORAGE_PROCESS_COUNT"] = "4"
    manifest = "\n".join(f"gs://dx-scin-public-data/dataset/images/{img}" for img in scin_needed)
    with open("/tmp/scin_manifest.txt", "w") as f:
        f.write(manifest)
    !cat /tmp/scin_manifest.txt | gcloud storage cp -I {LOCAL_IMAGE_DIR}/
    print("SCIN download complete")
else:
    print("SCIN: all present")

# ---- PAD-UFES: download zip from Mendeley ----
pad_ids = set(df[df["source"] == "pad-ufes"]["img_id"])
existing = {f.name for f in Path(LOCAL_IMAGE_DIR).iterdir() if f.is_file()}
pad_needed = pad_ids - existing
if pad_needed:
    print(f"\n--- PAD-UFES: downloading {len(pad_needed)} images from Mendeley ---")
    PAD_TMP = "/tmp/pad_ufes"
    os.makedirs(PAD_TMP, exist_ok=True)

    if not os.path.exists(f"{PAD_TMP}/done"):
        urls_to_try = [
            "https://data.mendeley.com/public-files/datasets/zr7vgbcyr2/files/757370e4-3a5a-4219-ae1c-cf1acfe4e22d/file_downloaded",
            "https://prod-dcd-datasets-cache-zipfiles.s3.eu-west-1.amazonaws.com/zr7vgbcyr2-1.zip",
        ]
        downloaded = False
        for url in urls_to_try:
            print(f"  Trying: {url[:80]}...")
            result = os.system(f'wget -q --show-progress --timeout=120 -O /tmp/pad_ufes.zip "{url}"')
            if result == 0 and os.path.exists("/tmp/pad_ufes.zip") and os.path.getsize("/tmp/pad_ufes.zip") > 1000000:
                print("  Download OK! Extracting...")
                !unzip -q -o /tmp/pad_ufes.zip -d {PAD_TMP}
                !touch {PAD_TMP}/done
                downloaded = True
                break
            else:
                print("  Failed, trying next...")
        if not downloaded:
            print("  ERROR: Could not auto-download PAD-UFES.")
            print("  Manual download: https://data.mendeley.com/datasets/zr7vgbcyr2/1")

    if os.path.exists(f"{PAD_TMP}/done"):
        pad_all = {f.name: f for f in Path(PAD_TMP).rglob("*.png")}
        print(f"  Found {len(pad_all)} .png files in archive")
        copied = 0
        for img_id in pad_needed:
            if img_id in pad_all:
                shutil.copy2(str(pad_all[img_id]), os.path.join(LOCAL_IMAGE_DIR, img_id))
                copied += 1
        print(f"  Copied {copied}/{len(pad_needed)} PAD-UFES images")
else:
    print("PAD-UFES: all present")

# ---- FINAL COUNT ----
existing = {f.name for f in Path(LOCAL_IMAGE_DIR).iterdir() if f.is_file()}
found = sum(1 for img_id in df["img_id"] if img_id in existing)
print(f"\n{'='*50}")
print(f"TOTAL: {found}/{len(df)} images ready")
for src in ["fitzpatrick17k", "scin", "pad-ufes"]:
    src_ids = set(df[df["source"] == src]["img_id"])
    src_found = len(src_ids & existing)
    print(f"  {src}: {src_found}/{len(src_ids)}")
if found < len(df):
    print(f"\n{len(df) - found} images could not be downloaded (dead URLs)")
print("="*50)

In [ ]:
# Cell 5: Augment minority classes (types 4, 5, 6)
from scripts.augment_minority import augment_images, verify_augmentation

# Only augment images we actually have
existing = {f.name for f in Path(LOCAL_IMAGE_DIR).iterdir() if f.is_file()}
df_available = df[df["img_id"].isin(existing)].copy()
print(f"Working with {len(df_available)}/{len(df)} available images\n")

all_augmented = []

for fitz_class, target_count in sorted(AUGMENTATION_TARGETS.items()):
    print(f"{'='*50}")
    print(f"Augmenting Fitzpatrick type {fitz_class}")
    print(f"{'='*50}")

    class_imgs = df_available[df_available["fitzpatrick_scale"] == fitz_class]["img_id"].tolist()
    print(f"Source images available: {len(class_imgs)}")
    print(f"Target new images: {target_count}")

    if not class_imgs:
        print(f"  SKIPPED — no source images available for type {fitz_class}")
        continue

    class_aug_dir = f"{LOCAL_AUG_DIR}/type_{fitz_class}"
    created, paths = augment_images(
        input_dir=LOCAL_IMAGE_DIR,
        output_dir=class_aug_dir,
        img_ids=class_imgs,
        target_count=target_count,
        seed=SEED + fitz_class,
        image_size=IMAGE_SIZE,
    )

    print(f"Created: {created} augmented images")

    result = verify_augmentation(LOCAL_IMAGE_DIR, class_aug_dir)
    print(f"Verification: {result['valid']}/{result['checked']} valid (passed={result['passed']})")
    assert result["passed"], f"Verification failed for type {fitz_class}: {result}"

    all_augmented.extend([
        {"img_id": p.name, "fitzpatrick_scale": fitz_class, "source": "augmented"}
        for p in paths
    ])

print(f"\nTotal augmented: {len(all_augmented)}")

In [ ]:
# Cell 6: Visual spot-check
import matplotlib.pyplot as plt
from PIL import Image as PILImage

fig, axes = plt.subplots(len(AUGMENTATION_TARGETS), 3, figsize=(12, 4 * len(AUGMENTATION_TARGETS)))

for row, fitz_class in enumerate(sorted(AUGMENTATION_TARGETS.keys())):
    class_aug_dir = Path(f"{LOCAL_AUG_DIR}/type_{fitz_class}")
    aug_files = sorted(class_aug_dir.glob("*"))[:3]
    for col, img_path in enumerate(aug_files):
        ax = axes[row][col]
        ax.imshow(PILImage.open(img_path))
        ax.set_title(f"Type {fitz_class}: {img_path.name}", fontsize=8)
        ax.axis("off")

plt.suptitle("Augmented Image Spot-Check (verify no artifacts)", fontsize=14)
plt.tight_layout()
plt.show()
print("Visually inspect the images above.")

In [ ]:
# Cell 7: Build balanced CSV (only from images we actually have)
aug_df = pd.DataFrame(all_augmented)
balanced_df = pd.concat([df_available, aug_df], ignore_index=True)

print("Balanced dataset:")
print(balanced_df["fitzpatrick_scale"].value_counts().sort_index())
print(f"\nTotal: {len(balanced_df)}")
print(f"  Original (available): {len(df_available)}")
print(f"  Augmented: {len(aug_df)}")

balanced_df.to_csv("balanced_dataset.csv", index=False)
print("\nSaved to balanced_dataset.csv")

In [ ]:
# Cell 8: Upload ALL images (originals + augmented) to GCS bucket
print("Uploading original images to bucket...")
os.environ["CLOUDSDK_STORAGE_THREAD_COUNT"] = "16"
os.environ["CLOUDSDK_STORAGE_PROCESS_COUNT"] = "4"
!gcloud storage cp -n {LOCAL_IMAGE_DIR}/* "{GCS_IMAGE_PREFIX}/"

print("\nUploading augmented images...")
for fitz_class in sorted(AUGMENTATION_TARGETS.keys()):
    class_aug_dir = f"{LOCAL_AUG_DIR}/type_{fitz_class}"
    if os.path.isdir(class_aug_dir):
        !gcloud storage cp -r {class_aug_dir}/* "{GCS_IMAGE_PREFIX}/"

!gcloud storage cp balanced_dataset.csv "gs://{BUCKET_NAME}/balanced_dataset.csv"
print("\nAll uploads complete!")

In [ ]:
# Cell 9: Generate AutoML manifest with stratified split
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    balanced_df, test_size=0.30, random_state=SEED,
    stratify=balanced_df["fitzpatrick_scale"],
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=SEED,
    stratify=temp_df["fitzpatrick_scale"],
)

train_df = train_df.copy(); train_df["split"] = "TRAINING"
val_df = val_df.copy(); val_df["split"] = "VALIDATION"
test_df = test_df.copy(); test_df["split"] = "TEST"

manifest_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

rows = []
for _, row in manifest_df.iterrows():
    gcs_path = f"{GCS_IMAGE_PREFIX}/{row['img_id']}"
    rows.append(f"{row['split']},{gcs_path},{int(row['fitzpatrick_scale'])}")

manifest_path = "data/automl_balanced_manifest.csv"
os.makedirs("data", exist_ok=True)
with open(manifest_path, "w", encoding="utf-8") as f:
    f.write("\n".join(rows) + "\n")

print(f"Manifest: {manifest_path} ({len(rows)} rows)")
print(f"  Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
print(f"\nPer-class in train:")
print(train_df["fitzpatrick_scale"].value_counts().sort_index())

!gcloud storage cp {manifest_path} "gs://{BUCKET_NAME}/automl/balanced_manifest.csv"
print("\nManifest uploaded!")

In [ ]:
# Cell 10: Run AutoML
from google.cloud import aiplatform

PROJECT_ID = "YOUR_PROJECT_ID"  # <-- FILL THIS IN
REGION = "us-central1"

aiplatform.init(project=PROJECT_ID, location=REGION)

dataset = aiplatform.ImageDataset.create(
    display_name="skin-tone-balanced",
    gcs_source=f"gs://{BUCKET_NAME}/automl/balanced_manifest.csv",
    import_schema_uri=aiplatform.schema.dataset.ioformat.image.single_label_classification,
)
print(f"Dataset created: {dataset.resource_name}")

job = aiplatform.AutoMLImageTrainingJob(
    display_name="skin-tone-balanced-automl",
    prediction_type="classification",
    multi_label=False,
    model_type="CLOUD",
)

model = job.run(
    dataset=dataset,
    model_display_name="skin-tone-balanced-v1",
    budget_milli_node_hours=8000,
)
print(f"Model trained: {model.resource_name}")

In [ ]:
# Cell 11: Evaluate
model_eval = model.list_model_evaluations()[0]
metrics = model_eval.metrics

print("AutoML Evaluation (Balanced Dataset):")
for key, value in metrics.items():
    print(f"  {key}: {value}")

if "confusionMatrix" in metrics:
    print("\nConfusion Matrix:")
    print(metrics["confusionMatrix"])

print(f"\nModel ID: {model.resource_name}")

In [ ]:
# Cell 12: Summary
print("=" * 60)
print("BALANCED AUGMENTATION SUMMARY")
print("=" * 60)
print(f"\nCSV total: {len(df)} images")
print(f"Available: {len(df_available)} images")
print(f"Augmented: {len(aug_df)}")
print(f"Balanced total: {len(balanced_df)}")
print(f"\nPer-class distribution:")
for cls in range(1, 7):
    orig = len(df_available[df_available["fitzpatrick_scale"] == cls])
    total = len(balanced_df[balanced_df["fitzpatrick_scale"] == cls])
    aug = total - orig
    print(f"  Type {cls}: {orig} original + {aug} augmented = {total}")
print(f"\nGCS: gs://{BUCKET_NAME}/")
print("=" * 60)